# V3/V4 to V5 migration guide

The legacy Epidata APIs, including the V4 main endpoint (`pub_covidcast()`)
and the V3 endpoints (`pub_fluview()`, `pub_flusurv()`, `pub_wiki()`, and so
on), are transitioning to the V5 API, served by `epidata_snapshot()`,
`epidata_archive()`, and `epidata_meta()`. The transition is happening source
by source. All V3 and V4 sources keep working until the migration is complete
(tentatively October 2026), and endpoints that are no longer updated remain
accessible on V3/V4. For new work, start on V5 and fall back to a legacy
method only for a source that is not yet available there.

Starting in October 2026, the V4 methods are tentatively deprecated in favor
of the V5 API, and calling them raises a `UserWarning` pointing back to this
guide.

For the current list of sources and indicators available on the new API, see
the [V5 signals documentation](https://cmu-delphi.github.io/delphi-epidata/api/v5_signals.html).

This guide walks through the transition from `pub_covidcast()` and the other
legacy methods. `pub_covidcast()` is the most widely used, but the V3 methods
differ in their names and argument conventions, so the tables below compare
both V4 (`pub_covidcast()`) and V3 (using `pub_fluview()` as the example) to
their V5 equivalents.

In [ ]:
# Hidden cell (set in the metadata for this cell)
import pandas as pd

# Set common options and context
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 1000)

In [ ]:
from epidatpy import EpiDataContext, EpiRange

epidata = EpiDataContext()

## Endpoint mapping

The legacy endpoints split into several purpose-built V5 routes determined by
the kind of query. The "V3 (other endpoints)" column gives examples
(`pub_fluview()`, `pub_flusurv()`, `pub_wiki()`) to illustrate how much the
legacy endpoints differ from each other; check each method's reference page
for its own behavior.

| Task | V4 (`pub_covidcast`) | V3 (other endpoints) | V5 equivalent |
|---|---|---|---|
| Fetch the latest data, or a snapshot as of a past date | `pub_covidcast()` (default, or with `as_of`) | Endpoint-specific (`pub_fluview()` has no `as_of`) | `epidata_snapshot()` |
| Fetch the full revision history of a signal | `pub_covidcast(issues=...)` | Supported by some (`pub_fluview()`, `pub_flusurv()` with `issues`) | `epidata_archive()` |
| Discover sources, signals, geo types, and date ranges | `pub_covidcast_meta()`, `CovidcastEpidata()` | Shared meta for some (`pub_fluview_meta()`) | `epidata_meta()` |
| Filter by publication lag | `pub_covidcast(lag=...)` | Supported by some (`pub_fluview()`, `pub_flusurv()`) | none (compute `report_time - reference_time`) |

`epidata()` is a convenience wrapper that routes to `epidata_archive()` if you
pass `report_time`, or to `epidata_snapshot()` if you pass `snapshot_date` (or
neither).

## Argument changes

Most `pub_covidcast()` arguments carry over to V5 under the same name, but
some have been renamed, dropped, or added. The V3 methods do not share
argument names with `pub_covidcast()`; `pub_fluview()` is shown as an example.

| V4 argument (`pub_covidcast`) | V3 (`pub_fluview`) | V5 argument | Notes |
|---|---|---|---|
| `data_source` | not exposed (implied by the method name) | `source` | Identifies the source dataset in V5. |
| `signals` | none (implied by the endpoint) | `signals` | One or more signal names within the source, sent in a single request. |
| `geo_type` | not exposed (`pub_fluview()` only has `regions`) | `geo_type` | Geographic resolution (e.g. `state`, `county`, `hhs`, `nation`). Accepts several values; one request is made per value. |
| `geo_values` | `regions` | `geo_values` | The server returns all locations for the requested `geo_type`; `geo_values` is filtered locally after the fetch. |
| `time_type` | not exposed (`pub_fluview()` is always weekly) | none | Dropped. All V5 endpoints use calendar dates. |
| `time_values` | `epiweeks` | `reference_time` | Accepts dates or an `EpiRange`. Filtered locally after the fetch. |
| `as_of` | none | `snapshot_date` | `epidata_snapshot()` only. A date, a `datetime`, or a UTC timestamp string such as `"2025-10-16T13:45:00Z"`. `None` returns the latest data. |
| `issues` | `issues` (where supported) | `report_time` | `epidata_archive()` only. A comparison string such as `"<2025-10-16"` or `">=2025-10-16T13:45:00Z"`, or an `EpiRange` for an inclusive range, both applied server-side. Bare dates and `"="` are rejected: use `snapshot_date` for point-in-time data. |
| `lag` | `lag` (where supported) | none | Compute it yourself from `epidata_archive()` output as `report_time - reference_time`; see [below](#lag). |
| none | none | `fill_method` | New in V5. Selects the imputation variant (`"source"`, `"fill_ave"`, or `"fill_zero"`); see [below](#fill-method). |
| none | none | `limit` | New in V5. Caps the rows returned, for previewing a query. The query has no stable sort order, so it is not a filter. |

(fill-method)=

The new methods also add `fill_method`, which has no covidcast equivalent.
Some sources publish several variants of the same signal that differ in how
nulls were handled during geographic aggregation:

- `"source"` is the raw source data, with no imputation
- `"fill_ave"` has null values filled with the average of neighboring values
- `"fill_zero"` has null values filled with zero

The default `None` returns all variants, so filter on this column (or pass a
value to the argument) if you want exactly one time series per location.

## Column changes

Response fields follow a similar pattern. `pub_fluview()` again serves as the
V3 example; column names vary across legacy endpoints (`pub_wiki()`, for
instance, returns `article`, `count`, and `hour`).

| V4 column (`pub_covidcast`) | V3 (`pub_fluview`) | V5 column | Notes |
|---|---|---|---|
| `source` | not returned | dropped | You queried by source; add it back with `.assign()` if you concatenate results across sources. |
| `signal` | none (implied by the endpoint) | `signal` | The signal name. |
| `value` | endpoint-specific columns (e.g. `num_ili`, `wili`, `ili`) | `value` | One standardized value column across all V5 sources. |
| not returned | not returned | `geo_type` | Included in V5 responses. |
| `geo_value` | `region` | `geo_value` | Standardized location identifier. |
| `time_value` | `epiweek` | `reference_time` | The date the value describes. Always a date. |
| `issue` | `issue` (where returned) | `report_time` | When the value was published. A UTC timestamp (`datetime64[ns, UTC]`), present in both snapshot and archive output. |
| `lag` | `lag` (where returned) | dropped | Compute as `report_time - reference_time`; see [calculating reporting lag](versioned_data.ipynb#calculating-reporting-lag). |
| `direction` | none | dropped | Was already deprecated in the covidcast API. |
| `stderr`, `sample_size` | none | `ci_lower`, `ci_upper` | Uncertainty is now expressed as confidence-interval bounds on `value`, for the sources that publish them. See [below](#uncertainty-columns). |
| `missing_value`, `missing_stderr`, `missing_sample_size` | none | dropped | Missingness is now expressed through `fill_method` variants and plain `NaN`s. |
| none | none | `fill_method` | Which null-handling variant of the signal this row belongs to. See [above](#fill-method). |

Some sources also carry extra columns in the new API, for example `age_group`
([pophive](https://cmu-delphi.github.io/delphi-epidata/api/v5-signals/epic-cosmos.html))
and `nwss_source`, `sample_index`, `pcr_target`
([nwss](https://cmu-delphi.github.io/delphi-epidata/api/v5-signals/nwss.html)).
Each source's documentation page lists its extra columns.

### Uncertainty columns

The covidcast columns `stderr` and `sample_size` have no fixed replacement.
The shared schema carries only `value`; a source that quantifies uncertainty
adds its own columns, such as `ci_lower` and `ci_upper`. Use the metadata or
the [documentation](https://cmu-delphi.github.io/delphi-epidata/api/v5_signals.html)
to see which value columns a source returns:

In [ ]:
meta_sleepcycle = epidata.epidata_meta(source="sleepcycle")
meta_sleepcycle["value_columns"]

## A query, before and after

### V4 example: NSSP through covidcast

Fetching NSSP influenza ED visit percentages for two states, as the data
looked on January 1, 2025:

In [ ]:
old = epidata.pub_covidcast(
    data_source="nssp",
    signals="pct_ed_visits_influenza",
    geo_type="state",
    time_type="week",
    geo_values=["pa", "ca"],
    time_values=EpiRange(202440, 202501),
    as_of=20250101,
).df()
old.head()

In [ ]:
new = epidata.epidata_snapshot(
    source="nssp",
    signals="pct_ed_visits_influenza",
    geo_type="state",
    geo_values=["pa", "ca"],
    reference_time=EpiRange("2024-10-01", "2025-01-01"),
    snapshot_date="2025-01-01",
).df()
new.head()

Both queries return the same signal, just with renamed and reshaped columns:

In [ ]:
print(list(old.columns))
print(list(new.columns))

### V3 example: FluView

For V3 endpoints like `pub_fluview()`, metrics that used to be separate
columns (`num_ili`, `ili`, `wili`, ...) become individual signal names that
you request through `signals`, and the results land in the single `value`
column:

In [ ]:
old_flu = epidata.pub_fluview(
    regions="nat",
    epiweeks=EpiRange(202440, 202445),
).df()
old_flu[["release_date", "region", "epiweek", "wili", "ili"]].head()

In [ ]:
new_flu = epidata.epidata_snapshot(
    source="fluview_ilinet",
    signals="wili",
    geo_type="nation",
    geo_values="us",
    reference_time=EpiRange("2024-10-01", "2024-11-15"),
).df()
new_flu.head()

## Revision history queries

Where you passed `issues` to `pub_covidcast()`, use `epidata_archive()` with
`report_time`:

In [ ]:
old_revisions = epidata.pub_covidcast(
    data_source="nssp",
    signals="pct_ed_visits_influenza",
    geo_type="state",
    time_type="week",
    geo_values="pa",
    time_values=EpiRange(202440, 202501),
    issues=EpiRange(202440, 202522),
).df()
old_revisions.head()

In [ ]:
revisions = epidata.epidata_archive(
    source="nssp",
    signals="pct_ed_visits_influenza",
    geo_type="state",
    geo_values="pa",
    reference_time=EpiRange("2024-10-01", "2025-01-01"),
    report_time="<2025-06-01",
).df()
revisions.head()

(lag)=

If you filtered by `lag`, fetch the archive and filter afterwards. Because
`report_time` is a timezone-aware timestamp and `reference_time` a naive date,
drop the timezone before subtracting:

In [ ]:
lag_days = (revisions["report_time"].dt.tz_localize(None) - revisions["reference_time"]).dt.days

# For an exact lag (e.g. 7 days):
revisions[lag_days == 7].head()

In [ ]:
# Or for maximum latency (e.g. at most 7 days of delay):
revisions[lag_days <= 7].head()

## Checking whether a source is available

Use `epidata_meta()` to see what a source offers in the new API. It returns
the signals, geo types, and the available `reference_time` and `report_time`
ranges:

In [ ]:
meta = epidata.epidata_meta(source="nssp")

# All the fields available for this source
list(meta)

In [ ]:
meta["signals"]  # available signal names

In [ ]:
meta["geo_types"]  # supported geography levels

In [ ]:
meta["reference_time_range"]  # earliest/latest reference_time available

In [ ]:
meta["report_time_range"]  # earliest/latest report_time (publication instant, UTC) available

Called with no argument, `epidata_meta()` returns the entries for every
available source, keyed by source name.

If `epidata_meta()` does not know the source yet, keep using
`pub_covidcast()` (or the relevant `pub_*`/`pvt_*` method) for it and check
back after package updates. The [API mailing
list](https://lists.andrew.cmu.edu/mailman/listinfo/delphi-covidcast-api)
announces sources as they move.

## Endpoints kept for historical reference

Not every V4 endpoint is moving to V5. The methods below cover data sources
whose collection has already ended (e.g. Google Flu Trends, the HealthTweets
signal, the various nowcasts). They are not part of the V4-to-V5 transition,
so they are not deprecated and will keep working. The historical data they
return is frozen and will remain available. They will just no longer receive
new data.

| Method | Data source |
|---|---|
| `pvt_cdc()` | CDC total and by-topic webpage visits |
| `pub_covid_hosp_facility_lookup()` | COVID hospitalization facility lookup |
| `pub_covid_hosp_facility()` | COVID hospitalizations by facility |
| `pub_covid_hosp_state_timeseries()` | COVID hospitalizations by state |
| `pub_delphi()` | Delphi's ILINet outpatient doctor visits forecasts |
| `pub_dengue_nowcast()` | Delphi's PAHO dengue nowcasts (Americas) |
| `pvt_dengue_sensors()` | PAHO dengue digital surveillance sensors (Americas) |
| `pub_ecdc_ili()` | ECDC ILI incidence (Europe) |
| `pub_gft()` | Google Flu Trends flu search volume |
| `pvt_ght()` | Google Health Trends health topics search volume |
| `pub_kcdc_ili()` | KCDC ILI incidence (Korea) |
| `pvt_meta_norostat()` | Metadata for the NoroSTAT endpoint |
| `pub_nidss_dengue()` | NIDSS dengue cases (Taiwan) |
| `pub_nidss_flu()` | NIDSS flu doctor visits (Taiwan) |
| `pvt_norostat()` | CDC NoroSTAT norovirus outbreaks |
| `pub_nowcast()` | Delphi's ILI Nearby nowcasts |
| `pub_paho_dengue()` | PAHO dengue data (Americas) |
| `pvt_sensors()` | Influenza and dengue digital surveillance sensors |
| `pvt_twitter()` | HealthTweets total and influenza-related tweets |
| `pub_wiki()` | Wikipedia webpage counts by article |